# Libraries

In [1]:
import re
import json
import unicodedata
from pathlib import Path
from typing import Optional
import pymupdf4llm

# Paths

In [2]:
BASE_DIR    = Path("../")
RAW_DIR     = BASE_DIR / "data" / "bronze"   
PROC_DIR    = BASE_DIR / "data" / "silver" 
META_DIR    = BASE_DIR / "data" / "metadata"

# Text Extraction

In [3]:
def extract_text_from_pdf(pdf_path: Path, paper_id: str) -> Optional[str]:
    """
    Extract clean text from a scientific PDF using PyMuPDF4LLM.
    
    PyMuPDF4LLM handles:
    - Multi-column layouts with correct reading order
    - Header/footer removal
    - Table extraction as Markdown
    - Image caption preservation
    """
    try:
        md_text = pymupdf4llm.to_markdown(str(pdf_path))
        return md_text
    except Exception as e:
        print(f"  ✗ Error extracting {paper_id}: {e}")
        return None

# Cleanning Functions

In [4]:
def remove_page_headers_footers(text: str) -> str:
    """
    Remove repetitive headers/footers.
    PyMuPDF4LLM already handles most, but we clean any remaining.
    """
    # arXiv headers - more aggressive pattern
    text = re.sub(r"arXiv:\d+\.\d+v\d+\s*\[[^\]]+\]\s*\d+\s+\w+\s+\d+\n?", "", text)
    # Standalone page numbers
    text = re.sub(r"^\s*\d{1,3}\s*$", "", text, flags=re.MULTILINE)
    # "Preprint" notices
    text = re.sub(r"Preprint\.\s+Under review[^\n]*\n?", "", text, flags=re.IGNORECASE)
    # Remove conference footer (e.g., '31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.')
    text = re.sub(r"\d+(?:st|nd|rd|th)\s+Conference on.*?USA\.\n?", "", text, flags=re.IGNORECASE)
    return text


def remove_footnotes(text: str) -> str:
    """
    Remove footnotes that appear after Abstract.
    Handles footnote markers like ∗, †, ‡
    """
    # Remove lines starting with footnote markers
    text = re.sub(r'^[∗†‡§¶]\s+.*$', '', text, flags=re.MULTILINE)
    # Remove footnote blocks
    text = re.sub(r'\n[∗†‡§¶].*?(?=\n\n|\n[A-Z]|\Z)', '', text, flags=re.DOTALL)
    return text


def remove_markdown_formatting(text: str) -> str:
    """
    Convert Markdown to plain text for RAG processing.
    """
    # Remove Markdown headers but keep the text
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    # Remove bold markers
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    # Remove italic markers
    text = re.sub(r'_(.+?)_', r'\1', text)
    text = re.sub(r'\*(.+?)\*', r'\1', text)
    # Remove code blocks (triple backticks)
    text = re.sub(r'```\n?(.*?)\n?```', r'\1', text, flags=re.DOTALL)
    # Remove inline code
    text = re.sub(r'`(.+?)`', r'\1', text)
    # Remove blockquotes
    text = re.sub(r'^>\s+', '', text, flags=re.MULTILINE)
    # Remove horizontal rules
    text = re.sub(r'^---+$', '', text, flags=re.MULTILINE)
    return text


def remove_paper_metadata(text: str) -> str:
    """
    Remove paper metadata that appears before the Abstract.
    Handles both plain text and Markdown formats.
    """
    # Try multiple patterns
    patterns = [
        r'## \*\*Abstract\*\*\s*\n',  # Markdown: ## **Abstract**
        r'## Abstract\s*\n',            # Markdown: ## Abstract
        r'#+\s*Abstract\s*\n',          # Any Markdown header with Abstract
        r'\*\*Abstract\*\*\s*\n',       # Bold: **Abstract**
        r'Abstract\s*\n',               # Plain: Abstract followed by newline
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return text[match.end():]
    
    return text


def normalize_whitespace(text: str) -> str:
    """
    Normalize whitespace:
    - Collapse multiple empty lines into at most one empty line
    - Remove trailing spaces on each line
    """
    lines = [line.rstrip() for line in text.split("\n")]
    cleaned = []
    prev_empty = False
    for line in lines:
        is_empty = len(line.strip()) == 0
        if is_empty and prev_empty:
            continue
        cleaned.append(line)
        prev_empty = is_empty
    return "\n".join(cleaned)


def remove_references_section(text: str) -> str:
    """
    Remove the References section at the end of the paper.
    Handles Markdown headers like '## **References**'.
    """
    ref_patterns = [
        r"^#+\\s*\\*\\*?References\\*\\*?\\s*$",  # ## References or ## **References**
        r"^\\*\\*?References\\*\\*?\\s*$",        # References or **References**
        r"^#+\\s*\\*\\*?Bibliography\\*\\*?\\s*$",
        r"^\\*\\*?Bibliography\\*\\*?\\s*$",
        r"^\\d+\\s*References\\s*$",
    ]

    for pattern in ref_patterns:
        match = re.search(pattern, text, re.MULTILINE | re.IGNORECASE)
        if match:
            if match.start() > len(text) * 0.75:
                return text[:match.start()].strip()

    return text


def clean_paper_text(raw_text: str) -> str:
    """
    Full cleaning pipeline for PyMuPDF4LLM Markdown output.
    """
    text = raw_text
    text = remove_page_headers_footers(text)
    text = remove_paper_metadata(text)
    text = remove_references_section(text)
    text = remove_markdown_formatting(text)
    text = remove_footnotes(text)
    text = normalize_whitespace(text)
    text = unicodedata.normalize("NFKC", text)
    return text.strip()

# Clean and Process Papers

In [5]:
corpus_stats = []

# Make sure output directories exist
PROC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Discover all PDFs in Bronze folder
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"✗ No PDF files found in {RAW_DIR}")
else:
    print(f"Found {len(pdf_files)} PDF files in {RAW_DIR}")

for pdf_path in pdf_files:
    paper_id = pdf_path.stem
    txt_path = PROC_DIR / f"{paper_id}.txt"

    # Extract raw text
    raw_text = extract_text_from_pdf(pdf_path, paper_id)
    if raw_text is None:
        continue

    # Clean
    clean_text = clean_paper_text(raw_text)
    if not clean_text.strip():
        print(f"  ✗ {paper_id}: Empty text after cleaning, skipping")
        continue

    # Save cleaned text
    txt_path.write_text(clean_text, encoding="utf-8")

    # Basic statistics
    char_count = len(clean_text)
    word_count = len(clean_text.split())
    para_count = len([p for p in clean_text.split("\n\n") if p.strip()])
    eq_count = clean_text.count("[EQ_START]")

    stats = {
        "id": paper_id,
        "pdf_file": pdf_path.name,
        "txt_file": txt_path.name,
        "chars": char_count,
        "words": word_count,
        "paragraphs": para_count,
        "equations": eq_count,
    }
    corpus_stats.append(stats)

    print(
        f"  ✓ {paper_id}: {word_count:,} words | "
        f"{para_count} paragraphs | {eq_count} equations"
    )

# Save corpus metadata
meta_path = META_DIR / "corpus_stats.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(corpus_stats, f, indent=2, ensure_ascii=False)

print(f"\n✓ Corpus processed. Stats saved to {meta_path}")
print(f"✓ Papers successfully processed: {len(corpus_stats)} / {len(pdf_files)}")

Found 10 PDF files in ../data/bronze
  ✓ 1502.04681v3: 8,268 words | 212 paragraphs | 0 equations


KeyboardInterrupt: 